In [1]:
"""
RF Uplift — Significance Tests Only
====================================
Loads the covariate CSV with all 9 treatments and tests whether
each treatment has a significant effect on comeback rate vs its control.
"""
import warnings
import pandas as pd
import statsmodels.api as sm

warnings.simplefilter("ignore")

FILE_PATH = r"Data/covariates_modeling_uplift_models_2026-03-13.csv"

TREATMENT_CONVERTER = {
    "BNLX_ChurnP_10_test_export.csv":       "treatment_1",
    "BNLX_ChurnP_10_controle_export.csv":    "control_1",
    "BNLX_ChurnP_25_test_export.csv":        "treatment_2",
    "BNLX_ChurnP_25_controle_export.csv":    "control_2",
    "BNLX_ChurnP_5eu_test_export.csv":       "treatment_3",
    "BNLX_ChurnP_5eu_controle_export.csv":   "control_3",
    "BNLX_ChurnP_10eu_test_export.csv":      "treatment_4",
    "BNLX_ChurnP_10eu_controle_export.csv":  "control_4",
    "BNLX_ChurnP_250_test_export.csv":       "treatment_5",
    "BNLX_ChurnP_250_controle_export.csv":   "control_5",
    "BNLX_ChurnP_500_test_export.csv":       "treatment_6",
    "BNLX_ChurnP_500_controle_export.csv":   "control_6",
    "BNLX_ChurnP_SKUe_test_export.csv":      "treatment_7",
    "BNLX_ChurnP_SKUe_controle_export.csv":  "control_7",
    "BNLX_ChurnP_SKUd_test_export.csv":      "treatment_8",
    "BNLX_ChurnP_SKUd_controle_export.csv":  "control_8",
    "BNLX_ChurnP_niks_test_export.csv":      "treatment_9",
    "BNLX_ChurnP_niks_controle_export.csv":  "control_9",
}

EXP_LABELS = {
    1: "10 pts", 2: "25 pts", 3: "€5 voucher", 4: "€10 voucher",
    5: "250 pts", 6: "500 pts", 7: "SKUe", 8: "SKUd", 9: "niks",
}


def load_data(path: str = FILE_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["treatment"] = df["treatment_indicator"].map(TREATMENT_CONVERTER)
    df = df.dropna(subset=["treatment"])
    df["experiment_k"] = df["treatment"].str.extract(r"(\d+)$").astype(int)
    return df


def test_reactivated_increase(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for exp_k, g in df.groupby("experiment_k"):
        ctrl = g.loc[g["treatment"].str.startswith("control"), "reactivated"].dropna()
        trt  = g.loc[g["treatment"].str.startswith("treatment"), "reactivated"].dropna()
        if ctrl.empty or trt.empty:
            continue

        rc, rt = ctrl.mean(), trt.mean()
        gs = pd.DataFrame({
            "reactivated": pd.concat([ctrl, trt]),
            "is_treat": [0] * len(ctrl) + [1] * len(trt),
        })

        model = sm.Logit(gs["reactivated"], sm.add_constant(gs["is_treat"])).fit(disp=0)
        coef, p2 = model.params["is_treat"], model.pvalues["is_treat"]

        # One-sided test when treatment > control
        p = (p2 / 2 if coef > 0 else 1 - p2 / 2) if rt > rc else p2
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

        results.append({
            "Incentive": EXP_LABELS.get(exp_k, str(exp_k)),
            "Reactivation_C": round(rc, 4),
            "Reactivation_T": round(rt, 4),
            "Uplift": round(rt - rc, 4),
            "p_value": round(p, 4),
            "sig": stars,
        })

    return pd.DataFrame(results).sort_values("p_value").reset_index(drop=True)


if __name__ == "__main__":
    df = load_data()
    print(f"Loaded {len(df):,} rows, {df['experiment_k'].nunique()} experiments\n")

    results = test_reactivated_increase(df)

    output_file = "Output/uplift_significance_results.xlsx"
    results.to_excel(output_file, index=False)
    print(f"Results saved to {output_file}")
    print(results)
    

Loaded 268,143 rows, 9 experiments

Results saved to Output/uplift_significance_results.xlsx
     Incentive  Reactivation_C  Reactivation_T  Uplift  p_value sig
0   €5 voucher          0.0205          0.0247  0.0042   0.0076  **
1      250 pts          0.0203          0.0236  0.0033   0.0249   *
2  €10 voucher          0.0204          0.0235  0.0031   0.0324   *
3      500 pts          0.0224          0.0252  0.0029   0.0514    
4       10 pts          0.0195          0.0216  0.0021   0.0978    
5       25 pts          0.0199          0.0220  0.0020   0.1085    
6         SKUe          0.0209          0.0223  0.0014   0.2043    
7         SKUd          0.0210          0.0217  0.0007   0.3404    
8         niks          0.0213          0.0213  0.0001   0.4853    
